# [8.1] Activation Patching Refresher - Solutions

Reference validation notebook for the section-local activation patching implementation.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter8_automated_circuits"
section = "part1_activation_patching_refresher"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_activation_patching_refresher.tests as tests
from chapter8_automated_circuits.exercises.part1_activation_patching_refresher import solutions

In [ ]:
tests.test_answer_logit_diff_validates_token_ids(solutions.answer_logit_diff)
tests.test_patch_activation_slice_replaces_one_component_without_mutating_inputs(
    solutions.patch_activation_slice,
)
tests.test_patching_recovery_report_and_sweep_normalize_by_clean_corrupt_gap(
    solutions.patching_recovery_report,
    solutions.activation_patching_sweep,
    solutions.recovery_fraction,
)
tests.test_localization_and_random_controls_require_top_components_to_win(
    solutions.patching_localization_report,
    solutions.random_patch_control_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["logit_diff"] == 2.0
assert contract["patch_slice"] == [[0.0, 0.0], [30.0, 40.0]]
assert contract["recovery"]["passes_recovery"]
assert contract["sweep"]["best_index"] == 1
assert contract["localization"]["localizes_target"]
assert contract["random_control"]["top_beats_random"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["sequence_length"] == 6
assert gpu["target_position"] == 5
assert gpu["best_position"] == 5
assert gpu["target_recovered_fraction"] == 1.0
assert gpu["passes_recovery"]
assert gpu["localizes_final_position"]
assert gpu["top_beats_wrong_position_control"]
assert gpu["max_abs_final_patch_logit_error"] <= 1e-5
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "model_name",
    "clean_metric",
    "corrupt_metric",
    "patched_metric",
    "target_recovered_fraction",
    "best_position",
    "wrong_position_control_fraction",
    "peak_vram_gb",
]}